# Shared analysis preparation

Check the processed data once and save its dates, source hashes and unresolved dispatch matches. Both modeling notebooks read this record.

## Setup

Use the same project folder and `project_settings.json` as the other notebooks.

In [ ]:
import sys
import json
import hashlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    import subprocess
    for package, module in [('numpy', 'numpy'), ('pandas', 'pandas'), ('pyarrow', 'pyarrow')]:
        if importlib.util.find_spec(module) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

if IN_COLAB:
    ROOT = Path("/content/drive/MyDrive/Milestone II")
else:
    ROOT = Path.cwd().resolve()

PROCESSED = ROOT / "data/processed"
SHARED = ROOT / "data/shared"

settings_path = ROOT / "project_settings.json"
if not settings_path.is_file():
    raise FileNotFoundError(f"Missing {settings_path}. Set PROJECT_FOLDER to your project folder.")

settings = json.loads(settings_path.read_text())

import numpy as np
import pandas as pd

STUDY_START = pd.Timestamp(settings["study_start"])
TRAIN_END = pd.Timestamp(settings["train_end"])
VAL_END = pd.Timestamp(settings["validation_end"])
TEST_END = pd.Timestamp(settings["test_end"])
PROFILE_AVAILABLE_FROM = pd.Timestamp(settings["forecast_profiles_available_from"])
CV_WINDOWS = settings["forward_validation_windows"]

assert STUDY_START < PROFILE_AVAILABLE_FROM < TRAIN_END < VAL_END < TEST_END
assert all(PROFILE_AVAILABLE_FROM < pd.Timestamp(a) < pd.Timestamp(b) <= VAL_END
           for a, b in CV_WINDOWS)

if settings["timezone"] != "America/New_York":
    raise ValueError("The prepared timestamps use Philadelphia local time. Rebuild data before changing timezone.")

missing = [name for name in ["trips_clean.parquet", "station_hour.parquet", "stations.parquet"]
           if not (PROCESSED / name).is_file()]

if missing:
    raise FileNotFoundError(f"Missing processed files: {missing}. Run 01 or copy the agreed processed data here.")

print("Project:", ROOT)
print(f"Training: {STUDY_START.date()} to {(TRAIN_END - pd.Timedelta(days=1)).date()}")
print(f"Validation: {TRAIN_END.date()} to {(VAL_END - pd.Timedelta(days=1)).date()}")
print(f"Test: {VAL_END.date()} to {(TEST_END - pd.Timedelta(days=1)).date()}")

Project: /content/drive/MyDrive/Milestone II
Training: 2024-01-01 to 2025-12-31
Validation: 2026-01-01 to 2026-03-31
Test: 2026-04-01 to 2026-06-30


In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import re
from difflib import SequenceMatcher
import numpy as np

def norm_name(value):
    """Ignore punctuation and spacing when comparing station names. Keep street numbers."""
    if pd.isna(value):
        return ""
    text = str(value).lower().replace("&", " and ")
    text = text.replace("m.l.k.", "martin luther king")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, content):
    Path(path).write_text(json.dumps(content, indent=2, default=str) + "\n")

In [ ]:
SHARED.mkdir(parents=True, exist_ok=True)


## Check the common inputs

Compare trip totals with station-hour departures, check keys and missing source values, and verify monthly coverage. The saved record identifies the actual files and the shared date settings. It contains no fitted model or clustering feature list.

In [ ]:
def prepare_shared(processed, shared):
    """Check the saved preparation outputs and record the common inputs."""
    processed, shared = Path(processed), Path(shared)
    shared.mkdir(parents=True, exist_ok=True)
    panel_columns = ["station_id", "hour", "departures", "temperature_c", "bikes_available"]
    panel = pd.read_parquet(processed / "station_hour.parquet", columns=panel_columns)
    stations = pd.read_parquet(processed / "stations.parquet")
    assert not stations.station_id.duplicated().any(), "Station IDs are not unique"
    assert not panel.duplicated(["station_id", "hour"]).any(), "Station-hour keys are not unique"
    assert panel.departures.ge(0).all(), "Departure counts cannot be negative"
    trip_count = pq.ParquetFile(processed / "trips_clean.parquet").metadata.num_rows
    assert panel.departures.sum() == trip_count, "Trips and panel departures differ"
    cleaning = pd.read_csv(processed / "cleaning_report.csv")
    assert cleaning.set_index("step").loc["rows_kept", "rows"] == trip_count
    panel["year_month"] = panel.hour.dt.to_period("M").astype(str)
    monthly = panel.groupby("year_month").agg(
        station_hours=("departures", "size"), departures=("departures", "sum"),
        stations=("station_id", "nunique"),
        weather_coverage=("temperature_c", lambda values: values.notna().mean()),
        status_coverage=("bikes_available", lambda values: values.notna().mean()),
    )
    reported = pd.read_csv(processed / "coverage_by_month.csv").set_index("year_month")
    assert monthly.index.equals(reported.index), "The monthly report has different months"
    for column in monthly:
        assert np.allclose(monthly[column], reported[column]), column
    quality = {"retained_trips": trip_count, "station_hours": len(panel),
               "panel_stations": int(panel.station_id.nunique()),
               "start": panel.hour.min(), "end": panel.hour.max(),
               "zero_share": panel.departures.eq(0).mean(),
               "status_coverage": panel.bikes_available.notna().mean()}
    del panel

    # Approximate dispatch matches cannot be repaired from the aggregated panel.
    # Save them for review, and tell forecasting which dispatch inputs to mask.
    dispatch_path = processed / "dispatch_name_matches.csv"
    uncertain_ids = []
    if dispatch_path.exists():
        matches = pd.read_csv(dispatch_path)
        review = matches.loc[~matches.method.isin(["normalized", "alias"])].copy()
        uncertain_ids = sorted(review.station_id.dropna().astype(int).unique().tolist())
    else:
        review = pd.DataFrame(columns=["dispatch_name", "station_id", "method"])
    review.to_csv(shared / "dispatch_review.csv", index=False)

    # Report source missingness before either analysis applies its own transformations.
    source_missing = []
    for name in ["trips_clean.parquet", "stations.parquet"]:
        totals = {}
        for batch in pq.ParquetFile(processed / name).iter_batches(batch_size=100_000):
            frame = batch.to_pandas()
            for column, count in frame.isna().sum().items():
                totals[column] = totals.get(column, 0) + int(count)
        total_rows = pq.ParquetFile(processed / name).metadata.num_rows
        source_missing.extend({"file": name, "column": col, "missing": value,
                               "rows": total_rows, "missing_share": value / total_rows}
                              for col, value in totals.items())
    pd.DataFrame(source_missing).to_csv(shared / "source_missingness.csv", index=False)
    pd.DataFrame([quality]).to_csv(shared / "data_quality.csv", index=False)
    sources = ["trips_clean.parquet", "station_hour.parquet", "stations.parquet",
               "cleaning_report.csv", "coverage_by_month.csv"]
    if dispatch_path.exists():
        sources.append(dispatch_path.name)
    manifest = {
        "version": 2, "sources": {name: file_hash(processed / name) for name in sources},
        "settings": settings,
        "uncertain_dispatch_station_ids": uncertain_ids,
        "dispatch_report_available": dispatch_path.exists(),
        "departure_target": "Unchanged departure counts in station_hour.parquet",
    }
    write_json(shared / "shared_manifest.json", manifest)
    return quality, review


quality, dispatch_review = prepare_shared(PROCESSED, SHARED)
print(f"Checked {quality['station_hours']:,} station-hours and {quality['retained_trips']:,} trips.")
display(pd.read_csv(SHARED / "source_missingness.csv").query("missing > 0"))


Checked 5,950,039 station-hours and 3,316,664 trips.


,file,column,missing,rows,missing_share
5,trips_clean.parquet,end_station_id,51217,3316664,0.015442
8,trips_clean.parquet,end_lat,51054,3316664,0.015393
9,trips_clean.parquet,end_lon,51054,3316664,0.015393
10,trips_clean.parquet,distance_km,51055,3316664,0.015393
12,trips_clean.parquet,passholder_type,1290,3316664,0.000389
21,stations.parquet,name,47,385,0.122078
22,stations.parquet,capacity,51,385,0.132468
23,stations.parquet,lat,39,385,0.101299
24,stations.parquet,lon,39,385,0.101299
25,stations.parquet,street,49,385,0.127273


## Review dispatch names

Check both saved name columns and keep street numbers. Similar names are review
candidates, not automatic replacements for the original matches.

In [ ]:
# Use the same name cleanup as data joining. Keep all street numbers.

def street_numbers(value):
    return set(re.findall(r"\b(\d+)(?:st|nd|rd|th)?\b", norm_name(value)))

stations = pd.read_parquet(PROCESSED / "stations.parquet")
catalogue = stations.melt(id_vars="station_id", value_vars=["station_name", "name"],
                         var_name="name_column", value_name="catalogue_name").dropna()
catalogue["clean_name"] = catalogue.catalogue_name.map(norm_name)
catalogue = catalogue.loc[catalogue.clean_name.ne("")].drop_duplicates()
station_lookup = stations.set_index("station_id")

evidence_rows, candidate_rows = [], []
for record in dispatch_review.itertuples(index=False):
    key = norm_name(record.dispatch_name)
    exact = catalogue.loc[catalogue.clean_name.eq(key)]
    exact_ids = sorted(exact.station_id.unique())
    suggested_id = exact_ids[0] if len(exact_ids) == 1 else pd.NA
    old_id = record.station_id
    old_name = station_lookup.at[int(old_id), "station_name"] if pd.notna(old_id) else ""
    left, right = street_numbers(key), street_numbers(old_name)
    number_conflict = bool(left and right and left != right)
    if len(exact_ids) > 1:
        finding = "Name belongs to several IDs; check dates and locations"
    elif len(exact_ids) == 1 and pd.notna(old_id) and suggested_id == old_id:
        finding = "Same ID supported by another saved station name"
    elif len(exact_ids) == 1:
        finding = "Exact saved name suggests a different or missing ID"
    else:
        finding = "No exact saved name; keep unresolved"
    evidence_rows.append({
        "dispatch_name": record.dispatch_name, "previous_id": old_id,
        "previous_station_name": old_name, "street_number_conflict": number_conflict,
        "suggested_id": suggested_id,
        "matched_name_columns": ", ".join(sorted(exact.name_column.unique())),
        "finding": finding,
        "action": "Check raw event dates, then correct data joining and rebuild dispatch aggregates",
    })

    # Similar names are candidates to inspect, never automatic replacements.
    ranked = catalogue.copy()
    ranked["similarity"] = ranked.clean_name.map(lambda name: SequenceMatcher(None, key, name).ratio())
    ranked = ranked.sort_values("similarity", ascending=False).drop_duplicates("station_id").head(3)
    for candidate in ranked.itertuples(index=False):
        candidate_rows.append({"dispatch_name": record.dispatch_name,
                               "candidate_id": candidate.station_id,
                               "catalogue_name": candidate.catalogue_name,
                               "name_column": candidate.name_column,
                               "similarity": candidate.similarity})

dispatch_evidence = pd.DataFrame(evidence_rows, columns=[
    "dispatch_name", "previous_id", "previous_station_name", "street_number_conflict",
    "suggested_id", "matched_name_columns", "finding", "action"])
dispatch_candidates = pd.DataFrame(candidate_rows, columns=[
    "dispatch_name", "candidate_id", "catalogue_name", "name_column", "similarity"])
dispatch_evidence.to_csv(SHARED / "dispatch_match_evidence.csv", index=False)
dispatch_candidates.to_csv(SHARED / "dispatch_match_candidates.csv", index=False)
display(dispatch_evidence.drop(columns="action"))
print("Candidate names and supporting evidence saved. No dispatch values have been changed.")

,dispatch_name,previous_id,previous_station_name,street_number_conflict,suggested_id,matched_name_columns,finding
0,26th & Moore - Vare Recreation Center,3253.0,27th & Morris - Vare Recreation Center,True,3253,name,Same ID supported by another saved station name
1,"26th & Moore, Vare Recreation Center",3253.0,27th & Morris - Vare Recreation Center,True,3253,name,Same ID supported by another saved station name
2,62nd & Market,3046.0,2nd & Market,True,3431,name,Exact saved name suggests a different or missi...
3,Aramingo & Cumberland,NaN,,False,3238,name,Exact saved name suggests a different or missi...


Candidate names and supporting evidence saved. No dispatch values have been changed.


## Check the reported coordinates over time

Use all cleaned departures for the IDs under review, including trips excluded from
clustering. Different coordinates may mean a move or a recording issue. First and last
trips are observed dates, not official opening or closing dates.

In [ ]:
review_ids = set(dispatch_evidence.previous_id.dropna().astype(int))
review_ids.update(dispatch_evidence.suggested_id.dropna().astype(int))
trip_parts = []
for batch in pq.ParquetFile(PROCESSED / "trips_clean.parquet").iter_batches(
        columns=["start_station_id", "start_time", "start_lat", "start_lon"]):
    trips = batch.to_pandas()
    trips = trips.loc[trips.start_station_id.isin(review_ids)].copy()
    if not trips.empty:
        trips["latitude"] = trips.start_lat.round(5)
        trips["longitude"] = trips.start_lon.round(5)
        trip_parts.append(trips)

location_columns = ["station_id", "latitude", "longitude", "departures", "first_departure", "last_departure"]
if trip_parts:
    reviewed_trips = pd.concat(trip_parts, ignore_index=True)
    missing_coords = reviewed_trips[["latitude", "longitude"]].isna().any(axis=1).sum()
    locations = reviewed_trips.groupby(["start_station_id", "latitude", "longitude"]).agg(
        departures=("start_time", "size"), first_departure=("start_time", "min"),
        last_departure=("start_time", "max")).reset_index().rename(columns={"start_station_id": "station_id"})
    locations = locations.sort_values(["station_id", "first_departure"])
    assert locations.departures.sum() + missing_coords == len(reviewed_trips)
    print(f"Checked {len(reviewed_trips):,} departures; {missing_coords:,} lack usable coordinates.")
    del reviewed_trips, trip_parts
else:
    locations = pd.DataFrame(columns=location_columns)
    print("No departures found for the IDs under review.")
locations.to_csv(SHARED / "dispatch_station_location_history.csv", index=False)
display(locations)
print("Several locations under one ID need date-specific review before adding an alias.")

Checked 34,344 departures; 0 lack usable coordinates.


,station_id,latitude,longitude,departures,first_departure,last_departure
0,3046,39.95012,-75.14472,12827,2024-01-01 00:21:00,2025-03-18 09:48:00
1,3238,39.94628,-75.15138,16883,2024-01-01 00:45:00,2025-12-31 22:31:00
2,3238,39.97674,-75.11995,362,2026-01-01 01:30:00,2026-06-30 09:34:00
4,3253,39.93174,-75.18996,1517,2024-01-01 09:13:00,2024-12-31 09:47:00
3,3253,39.93074,-75.18924,2205,2025-01-01 00:05:00,2026-06-30 23:10:00
5,3431,39.96239,-75.24522,550,2025-07-20 09:17:00,2026-06-30 09:17:00


Several locations under one ID need date-specific review before adding an alias.


## Check raw event dates, if available

Compare flagged endpoints with the trip history. Missing event times stay missing.
Correct confirmed mappings in data joining, then rebuild the panel; editing a review CSV
does not fix the saved counts.

In [ ]:
raw_paths = sorted((ROOT / "data/raw/dispatch70/dispatch").glob("*.parquet"))
event_parts = []
evidence_lookup = dispatch_evidence.assign(
    clean_name=dispatch_evidence.dispatch_name.map(norm_name)).drop_duplicates("clean_name").set_index("clean_name")

for path in raw_paths:
    for batch in pq.ParquetFile(path).iter_batches(columns=[
            "from_station", "to_station", "t_from", "t_to", "kind", "n_bikes"]):
        raw = batch.to_pandas()
        for side in ["from", "to"]:
            keys = raw[f"{side}_station"].map(norm_name)
            selected = raw.loc[keys.isin(evidence_lookup.index)].copy()
            if selected.empty:
                continue
            selected["dispatch_name"] = selected[f"{side}_station"]
            selected["side"] = side
            # Use this endpoint's time, with the other endpoint as a recorded fallback.
            other = "to" if side == "from" else "from"
            stamp = pd.to_numeric(selected[f"t_{side}"], errors="coerce").fillna(
                pd.to_numeric(selected[f"t_{other}"], errors="coerce"))
            selected["event_time"] = pd.to_datetime(stamp, unit="s", utc=True, errors="coerce").dt.tz_convert(
                "America/New_York").dt.tz_localize(None)
            selected["suggested_id"] = selected.dispatch_name.map(norm_name).map(evidence_lookup.suggested_id)
            selected["source_file"] = path.name
            event_parts.append(selected[["dispatch_name", "side", "event_time", "kind", "n_bikes",
                                         "suggested_id", "source_file"]])

if event_parts:
    event_evidence = pd.concat(event_parts, ignore_index=True)
    active = locations.groupby("station_id").agg(
        first_observed=("first_departure", "min"), last_observed=("last_departure", "max"))
    event_evidence = event_evidence.join(active, on="suggested_id")
    known = event_evidence[["event_time", "first_observed", "last_observed"]].notna().all(axis=1)
    event_evidence["outside_observed_trip_dates"] = pd.Series(pd.NA, index=event_evidence.index, dtype="boolean")
    event_evidence.loc[known, "outside_observed_trip_dates"] = (
        event_evidence.loc[known, "event_time"].lt(event_evidence.loc[known, "first_observed"])
        | event_evidence.loc[known, "event_time"].gt(event_evidence.loc[known, "last_observed"]))
    event_evidence.to_csv(SHARED / "dispatch_event_evidence.csv", index=False)
    display(event_evidence.groupby(["dispatch_name", "suggested_id"], dropna=False).agg(
        endpoint_records=("side", "size"), first_event=("event_time", "min"), last_event=("event_time", "max"),
        dates_outside_observed_trips=("outside_observed_trip_dates", "sum")))
    print("Saved individual event endpoints. Compare their dates with each location's history above.")
else:
    # Remove an old report so an earlier run is not mistaken for this run's result.
    (SHARED / "dispatch_event_evidence.csv").unlink(missing_ok=True)
    print("Raw dispatch files are absent." if not raw_paths else "No flagged names found in the available raw files.")
    print("The event-date check is incomplete. Keep the dispatch restriction until the source is checked.")

,,endpoint_records,first_event,last_event,dates_outside_observed_trips
dispatch_name,suggested_id,,,,
26th & Moore - Vare Recreation Center,3253,7,2025-04-15 16:30:00,2025-04-20 16:00:00,0
"26th & Moore, Vare Recreation Center",3253,93,2025-04-22 10:15:00,2026-06-29 17:45:00,0
62nd & Market,3431,16,2025-12-19 17:15:00,2026-05-03 17:30:00,0
Aramingo & Cumberland,3238,1,2026-06-20 15:15:00,2026-06-20 15:15:00,0


Saved individual event endpoints. Compare their dates with each location's history above.
